# Visualization — Evaluation Report

**How to run:**
1. Run `005_run_batch_inference_colab.ipynb` first to generate `rows_cache_*.json`.
2. Set `CACHE_FILE` in Section 1 to point to that file.
3. `Runtime → Run all`.

## 0 · Install Dependencies

> After installation → **Restart session** before continuing.

In [ ]:
!pip install Levenshtein==0.26.1 plotnine==0.14.5 evaluate==0.4.4 cer==1.2.0 rouge_score==0.1.2 seaborn bitsandbytes python-dateutil --quiet

## 1 · Configuration

In [ ]:
import os, json
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_HF'] = '1'

BASE_DIR = Path('/content/drive/MyDrive/INTERN-BIWOCO/sample-for-multi-modal-document-to-json-with-sagemaker-ai')

def is_valid_checkpoint(ckpt_path):
    cfg = ckpt_path / 'adapter_config.json'
    if not cfg.exists(): return False
    data = json.load(open(cfg))
    return 'language_model' not in data.get('target_modules', '')

CHECKPOINT = str(sorted(
    [p for p in (BASE_DIR / 'models/finetune').glob('*/checkpoint-*')
     if is_valid_checkpoint(p)],
    key=lambda p: p.stat().st_mtime
)[-1])

DATASET_DIR = BASE_DIR / 'data' / 'swift_dataset'
IMAGES_DIR  = DATASET_DIR / 'images'
MODEL_NAME  = Path(CHECKPOINT).parent.name
CACHE_FILE  = BASE_DIR / 'rows_cache.json'  # ← point to your cache file

print(f"BASE_DIR   : {BASE_DIR}")
print(f"CHECKPOINT : {CHECKPOINT}")
print(f"MODEL_NAME : {MODEL_NAME}")
print(f"Cache      : {'EXISTS' if CACHE_FILE.exists() else 'NOT found — will run inference'}")

In [ ]:
# Clear VRAM before inference (essential after training on T4 15GB)
import torch, gc

for var in ['model', 'trainer', 'engine', 'optimizer']:
    if var in dir():
        del globals()[var]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.mem_get_info()[1] / 1024**3
print(f'VRAM free: {free:.1f} GB / {total:.1f} GB')
if free < 8:
    print('WARNING: < 8 GB free — inference may OOM. Try Runtime > Restart & Run All.')

## 2 · Load Data

In [ ]:
import json, torch, gc
import numpy as np
from pathlib import Path

HINTS_BY_DOCTYPE = {
    "AUS_DRIVER_LICENSE": (
        "Pay attention to:\n"
        "- conditions_legend: dict of single-letter codes mapped to descriptions\n"
        "- front and back are separate nested objects\n"
        "- dob_watermark: DDMMYYYY format"
    ),
    "AUS_PASSPORT": (
        "Pay attention to:\n"
        "- mrz_line1 and mrz_line2: exactly 44 characters each, at the bottom of the passport\n"
        "- Include ALL '<' characters in MRZ lines, do not omit any\n"
        "- mrz_line1 starts with P<AUS\n"
        "- mrz_line2 starts with the document number"
    ),
    "AUS_MEDICARE_CARD": (
        "Pay attention to:\n"
        "- cardholders is a list of objects with position, first_name, middle_initial, last_name, full_name\n"
        "- card_number format: XXXX XXXXX X\n"
        "- expiry_date format: YYYY-MM-DD"
    ),
    "AUS_ENERGY_BILL": (
        "Pay attention to:\n"
        "- all monetary amounts are float (e.g. 928.80)\n"
        "- billing_days is integer\n"
        "- electricity_kwh and gas_mj can have decimals\n"
        "- null for fields not present on the bill"
    ),
    "AUS_WWC_CARD": (
        "Pay attention to:\n"
        "- wwc_type is either 'Employee' or 'Volunteer'\n"
        "- expiry_date format YYYY-MM-DD"
    ),
}
DEFAULT_HINT = "Extract all fields carefully. Return null for any field not visible."

def _safe(v):
    if v is None: return ''
    if isinstance(v, list): return json.dumps(v, ensure_ascii=False)
    return str(v).strip()

def build_schema(d: dict) -> dict:
    return {k: build_schema(v) if isinstance(v, dict) else ([] if isinstance(v, list) else None)
            for k, v in d.items()}

def load_or_infer():
    if CACHE_FILE.exists():
        print(f"Loading from cache: {CACHE_FILE}")
        with open(CACHE_FILE) as f:
            return json.load(f)

    print("No cache found — running inference...")
    from swift import TransformersEngine, RequestConfig, InferRequest
    from tqdm import tqdm

    with open(DATASET_DIR / 'conversations_test_swift_format.json') as f:
        test_data = json.load(f)
    with open(DATASET_DIR / 'conversations_train_swift_format.json') as f:
        train_data = json.load(f)

    schema_by_doctype = {}
    for s in train_data:
        gt = json.loads(s['messages'][2]['content'])
        dt = gt.get('document_type', '')
        if dt not in schema_by_doctype:
            schema_by_doctype[dt] = build_schema(gt)

    engine  = TransformersEngine('Qwen/Qwen2-VL-2B-Instruct', adapters=[CHECKPOINT],
                                  max_pixels=150528, quantization_bit=4,
                                  torch_dtype='bfloat16', max_length=1024)

    req_cfg = RequestConfig(max_tokens=1024, temperature=0, logprobs=True)

    def predict(sample):
        imgs     = [str(IMAGES_DIR / Path(p).name) for p in sample['images']]
        gt       = json.loads(sample['messages'][2]['content'])
        doc_type = gt.get('document_type', '')
        schema   = json.dumps(schema_by_doctype.get(doc_type, {}), indent=2, ensure_ascii=False)
        prompt   = (
            f"Extract all fields from this {doc_type} document.\n"
            f"{HINTS_BY_DOCTYPE.get(doc_type, DEFAULT_HINT)}\n\n"
            f"Return ONLY valid JSON matching this structure (keys must match exactly):\n"
            f"{schema}\n\n"
            f"Rules:\n- null for missing fields, do NOT omit keys\n- Dates: YYYY-MM-DD\n- No markdown, no explanation"
        )
        msgs   = [sample['messages'][0], {'role': 'user', 'content': prompt}]
        result = engine.infer([InferRequest(messages=msgs, images=imgs)], req_cfg)[0]
        out    = result.choices[0].message.content.strip()
        out    = out.strip('`').removeprefix('json').strip()

        # ── confidence ──────────────────────────────────────────────────
        # logprobs: SWIFT may return dict or object
        _lp = result.choices[0].logprobs
        if isinstance(_lp, dict): _content = _lp.get("content") or []
        elif _lp is not None:     _content = _lp.content or []
        else:                     _content = []
        token_logprobs = [t["logprob"] if isinstance(t, dict) else t.logprob for t in _content]
        avg_confidence  = float(np.exp(np.mean(token_logprobs))) if token_logprobs else None

        try:
            pred = json.loads(out)
            flat = flatten_dict(pred)
            n    = len(flat)
            chunks = np.array_split(token_logprobs, n) if n > 0 and token_logprobs else []
            field_confidences = {
                key: float(np.exp(np.mean(chunk))) if len(chunk) > 0 else None
                for (key, _), chunk in zip(flat.items(), chunks)
            }
            return pred, True, avg_confidence, field_confidences
        except:
            return {}, False, avg_confidence, {}

    rows = []
    for s in tqdm(test_data, desc='Inference'):
        gt       = json.loads(s['messages'][2]['content'])
        pred, ok, avg_conf, field_conf = predict(s) 
        rows.append({
            'image': s['images'][0],
            'doc_type': gt.get('document_type'),
            'gt': gt,
            'pred': pred,
            'valid_json': ok,
            'confidence' : avg_conf,       
            'field_confidences': field_conf, 
            
            })

    del engine; torch.cuda.empty_cache(); gc.collect()

    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"Cache saved to {CACHE_FILE}")
    return rows

rows = load_or_infer()
print(f"\nTotal samples : {len(rows)}")
print(f"Valid JSON    : {sum(r['valid_json'] for r in rows)}/{len(rows)}")

## 2.5 · Normalize Dates to YYYY-MM-DD

Converts all date values in both `gt` and `pred` to `YYYY-MM-DD` before any metric calculation.

Handles formats such as: `DD/MM/YYYY`, `MM/DD/YYYY`, `DD-MM-YYYY`, `YYYY/MM/DD`, `D MMM YYYY`, `YYYYMMDD`, etc.

In [ ]:
from dateutil import parser as dtparser
from dateutil.parser import ParserError
from datetime import datetime
import re

DATE_KEYWORDS = ['date', 'ngay', 'day', 'dated', 'dob', 'birth', 'expiry', 'expire',
                 'issued', 'issue', 'valid', 'from', 'to', 'start', 'end', 'period']

def is_date_field(field_name: str) -> bool:
    return any(kw in field_name.lower() for kw in DATE_KEYWORDS)

def try_parse_date(value: str) -> str:
    if not value or not isinstance(value, str):
        return value
    v = value.strip()
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}', v):
        return v
    patterns = [
        (r'^(\d{1,2})/(\d{1,2})/(\d{4})$', '%d/%m/%Y'),
        (r'^(\d{4})/(\d{2})/(\d{2})$',      '%Y/%m/%d'),
        (r'^(\d{1,2})-(\d{1,2})-(\d{4})$',  '%d-%m-%Y'),
        (r'^(\d{8})$',                        '%Y%m%d'),
    ]
    for pattern, fmt in patterns:
        if re.fullmatch(pattern, v):
            try: return datetime.strptime(v, fmt).strftime('%Y-%m-%d')
            except ValueError: pass
    try: return dtparser.parse(v, dayfirst=True).strftime('%Y-%m-%d')
    except (ParserError, OverflowError, ValueError): return v

def normalize_dates(record: dict) -> dict:
    for k, v in record.items():
        if is_date_field(k) and isinstance(v, str):
            record[k] = try_parse_date(v)
    return record

n_converted = 0
for r in rows:
    before = (json.dumps(r['gt'], sort_keys=True), json.dumps(r['pred'], sort_keys=True))
    normalize_dates(r['gt'])
    normalize_dates(r['pred'])
    if (json.dumps(r['gt'], sort_keys=True), json.dumps(r['pred'], sort_keys=True)) != before:
        n_converted += 1

print(f"Date normalization complete. Rows changed: {n_converted} / {len(rows)}")
date_fields = [f for r in rows for f in r['gt'] if is_date_field(f)]
print(f"Date fields detected: {sorted(set(date_fields))}")

## 3 · Build DataFrame

`dist` encoding:

| Value | Meaning |
|---|---|
| -1 | missing groundtruth |
| 0..6 | Levenshtein distance |
| -2 → 7 | predicted 'None' |
| -3 → 8 | key missing |
| -4 → 9 | invalid JSON |

In [ ]:
import pandas as pd
import Levenshtein

def flatten_dict(d, parent_key='', sep='.'):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        elif isinstance(v, list):
            items[new_key] = json.dumps(v, ensure_ascii=False)
        else:
            items[new_key] = v
    return items

EXCLUDE_FIELDS = {'back.barcode_number'}

for r in rows:
    r['gt_flat']   = flatten_dict(r['gt'])
    r['pred_flat'] = flatten_dict(r['pred']) if r['valid_json'] else {}

all_fields = sorted({k for r in rows for k in r['gt_flat']} - EXCLUDE_FIELDS)
records = []

for r in rows:
    for field in all_fields:
        if field not in r['gt_flat']:
            continue
        gt_val = _safe(r['gt_flat'].get(field))

        if not r['valid_json']:
            pred_val, dist = '', -4
        elif field not in r['pred_flat']:
            pred_val, dist = '', -3
        elif gt_val == '':
            pred_val = _safe(r['pred_flat'].get(field, ''))
            dist = -1
        else:
            pred_val = _safe(r['pred_flat'].get(field, ''))
            dist = -2 if pred_val in ('None', '') else Levenshtein.distance(pred_val, gt_val)

        records.append({
            'image'       : r['image'],
            'doc_type'    : r['doc_type'],
            'entity'      : field,
            'label_val'   : gt_val,
            'response_val': pred_val,
            'dist'        : dist,
            'valid_json'  : r['valid_json'],
            'model'       : CHECKPOINT,
            'pretty_name' : MODEL_NAME,
            'confidence'  : r.get('field_confidences', {}).get(field),
        })

df_multi = pd.DataFrame(records)
print(f"df_multi shape: {df_multi.shape}")
df_multi.head(3)

In [ ]:
# Mean confidence per sample (overall)
conf_df = pd.DataFrame([
    {'image': r['image'], 'doc_type': r['doc_type'], 'confidence': r.get('confidence')}
    for r in rows if r.get('confidence') is not None
])
print("Average confidence per doc_type:")
print(conf_df.groupby('doc_type')['confidence'].mean().round(4))

## 4 · Feature Categorization

### 4.1 · Null Percentage by Entity

In [ ]:
from plotnine import ggplot, aes, geom_bar, theme_bw, labs, coord_flip

null_df = (
    df_multi.groupby('entity')['label_val']
    .apply(lambda x: (x == '').sum() / len(x) * 100)
    .reset_index()
    .rename(columns={'label_val': 'metric_value'})
    .sort_values('metric_value', ascending=True)
)
null_df['entity'] = pd.Categorical(null_df['entity'], categories=null_df['entity'].tolist(), ordered=True)

(ggplot(null_df, aes(x='entity', y='metric_value')) +
 geom_bar(stat='identity', fill='steelblue') +
 coord_flip() +
 labs(title='Null Percentage by Entity', x='Entity', y='Null Percentage (%)') +
 theme_bw())

### 4.2 · Entity String Length — Boxplot

In [ ]:
from plotnine import geom_boxplot, theme_minimal, theme

df_len = df_multi[df_multi['label_val'] != ''].copy()
df_len['str_len'] = df_len['label_val'].str.len()

entity_order = (
    df_len.groupby('entity')['str_len']
    .mean().sort_values().index.tolist()
)
df_len['entity'] = pd.Categorical(df_len['entity'], categories=entity_order, ordered=True)

(ggplot(df_len, aes(x='entity', y='str_len')) +
 geom_boxplot(fill='#abd9e9', color='#2c7bb6', outlier_alpha=0.3) +
 coord_flip() +
 labs(title='String Length Distribution by Entity', x='Entity', y='String Length') +
 theme_minimal() +
 theme(figure_size=(10, 6)))

### 4.3 · Categorize Features

In [ ]:
from enum import Enum

NULL_THRESHOLD   = 70
LENGTH_THRESHOLD = 50

class FeatureCategory(str, Enum):
    MISSING_GROUND_TRUTH = 'missing_ground_truth'
    SHORT_TEXT           = 'short_text'
    LONG_TEXT            = 'long_text'

stats_records = []
for entity, group in df_multi.groupby('entity'):
    labels   = group['label_val']
    null_pct = (labels == '').sum() / len(labels) * 100
    non_null = labels[labels != '']
    len_mean = non_null.str.len().mean() if len(non_null) > 0 else 0.0
    stats_records.append({'entity': entity, 'null_pct': null_pct, 'len_mean': len_mean})

entity_stats = pd.DataFrame(stats_records)

def categorize(row):
    if row['null_pct'] > NULL_THRESHOLD:    return FeatureCategory.MISSING_GROUND_TRUTH
    elif row['len_mean'] <= LENGTH_THRESHOLD: return FeatureCategory.SHORT_TEXT
    else:                                     return FeatureCategory.LONG_TEXT

entity_stats['category'] = entity_stats.apply(categorize, axis=1)
feature_categories = dict(zip(entity_stats['entity'], entity_stats['category']))

for cat in FeatureCategory:
    members = [e for e, c in feature_categories.items() if c == cat]
    print(f"\n{cat.value} ({len(members)}): {members}")

## 5 · Edit Distance Heatmap

In [ ]:
from plotnine import *

df = df_multi.copy()
df["dist_cut"] = df["dist"].clip(upper=6)
df["dist_cut"] = df["dist_cut"].replace({-2: 7, -3: 8, -4: 9})

mapper = {
    -1: "missing groundtruth",
     0: "exact match",
     1: "1", 2: "2", 3: "3", 4: "4", 5: "5", 6: "6+",
     7: "predicted 'None'",
     8: "key missing",
     9: "invalid JSON"
}

df["dist_cut"]    = pd.Categorical(df["dist_cut"].replace(mapper), categories=list(mapper.values()), ordered=True)
df["entity_type"] = df["entity"].map(feature_categories)
df["entity_label"] = df["doc_type"].fillna("UNKNOWN") + " | " + df["entity"]

df_sorted    = df.sort_values(["doc_type", "entity_type", "entity"])
df_entities  = df_sorted[["entity_label", "entity_type", "doc_type"]].drop_duplicates(subset=["entity_label"]).reset_index(drop=True)
ordered_values = list(df_entities["entity_label"])

change_indices = df_entities.index[
    (df_entities["entity_type"] != df_entities["entity_type"].shift()) |
    (df_entities["doc_type"]    != df_entities["doc_type"].shift())
].tolist()
feature_class_lines = [i + 0.5 for i in change_indices[1:]]

df_sorted['pretty_name_ordered'] = pd.Categorical(
    df_sorted['pretty_name'],
    categories=sorted(df_sorted['pretty_name'].unique()),
    ordered=True
)

def plot_bar_chart_all_entities(df, ordered_values, width=22, height=18, geom_vlines=[]):
    custom_colors = ["#bababa","#66c2a5","#abdda4","#e6f598","#ffffbf",
                     "#fee08b","#fdae61","#f46d43","#abd9e9","#74add1","#bebada"]
    return (
        ggplot(df, aes(x="entity_label", fill="factor(dist_cut)"))
        + geom_bar(position="stack", color="black")
        + facet_wrap("~pretty_name_ordered", scales="free")
        + labs(x="Doc Type | Field", y="Count", fill="Char. edit distance")
        + coord_flip()
        + scale_fill_manual(values=custom_colors, labels=list(mapper.values()))
        + theme_minimal()
        + theme(figure_size=(width, height))
        + scale_x_discrete(limits=ordered_values)
    )

In [ ]:
from IPython.display import Markdown, display

plot = plot_bar_chart_all_entities(df_sorted, ordered_values, width=15, height=14, geom_vlines=feature_class_lines)
display(Markdown("### All Entities"))
plot.show()

## 6 · Model Performance Metrics (Exact Match, CER, BLEU, ROUGE)

In [ ]:
import evaluate

eval_metrics = ["exact_match", "character", "bleu", "rouge"]
evaluations  = [evaluate.load(m) for m in eval_metrics]

def calculate_metrics(references, predictions):
    pairs = [(r, p) for r, p in zip(references, predictions) if r.strip() != ""]
    if not pairs:
        return pd.Series({'exact_match': 0.0, 'cer': None, 'bleu': None, 'rouge': None})
    refs, preds = zip(*pairs)
    results = {}
    for metric in evaluations:
        results.update(metric.compute(predictions=list(preds), references=list(refs)))
    return results

In [ ]:
relevant_entities = [
    e for e, cat in feature_categories.items()
    if cat != FeatureCategory.MISSING_GROUND_TRUTH
]
df_filtered = df_multi[df_multi['entity'].isin(relevant_entities)].copy()

df_eval = (
    df_filtered
    .groupby(['model', 'pretty_name', 'entity'])
    .apply(
        lambda x: calculate_metrics(
            x['label_val'].astype(str).tolist(),
            x['response_val'].astype(str).tolist()
        ),
        include_groups=False
    )
    .reset_index()
)
df_eval = pd.concat([df_eval, df_eval[0].apply(pd.Series)], axis=1).drop(0, axis=1)
print(f'df_eval shape: {df_eval.shape}')
df_eval.head(3)

### 6.1 · Exact Match per Entity

In [ ]:
from plotnine import *

ordered_relevant = [e for e in relevant_entities]

df_entities_rel = df_entities[df_entities['entity_label'].apply(
    lambda x: x.split(' | ')[-1] if ' | ' in x else x
).isin(relevant_entities)].reset_index(drop=True)

change_idx_rel = df_entities_rel.index[
    df_entities_rel['entity_type'] != df_entities_rel['entity_type'].shift()
].tolist()
feat_lines_rel = [i + 0.5 for i in change_idx_rel[1:]]

(
    ggplot(df_eval, aes(x='entity', y='exact_match', fill='pretty_name'))
    + geom_bar(stat='identity', position='dodge', colour='gray')
    + labs(title='Exact Match per Entity (higher = better)', x='Entity', y='Exact Match', fill='Model')
    + coord_flip()
    + scale_fill_brewer(type='qual', palette='Set3')
    + theme_minimal()
    + theme(figure_size=(12, 8), axis_text_x=element_text(angle=45, hjust=1))
    + scale_y_continuous(
        breaks=[x/100 for x in range(0, 101, 10)],
        labels=[f'{i}%' for i in range(0, 101, 10)]
    )
    + geom_hline(yintercept=[x/100 for x in range(0, 101, 10)], color='darkgray', size=0.4, alpha=0.5)
    + scale_x_discrete(limits=ordered_relevant)
)

### 6.2 · Character Error Rate (CER) per Entity

In [ ]:
(
    ggplot(df_eval, aes(x='entity', y='cer_score', fill='pretty_name'))
    + geom_bar(stat='identity', position='dodge', color='gray')
    + labs(title='CER per Entity (lower = better)', x='Entity', y='CER', fill='Model')
    + coord_flip()
    + scale_fill_brewer(type='qual', palette='Set3')
    + theme_minimal()
    + theme(figure_size=(12, 8), axis_text_x=element_text(angle=45, hjust=1))
    + geom_vline(xintercept=feat_lines_rel, linetype='dashed', color='#4d4d4d', size=2.0)
    + scale_x_discrete(limits=ordered_relevant)
)

### 6.3 · Aggregate Metrics Table

- **Short text** entities → Exact Match + CER  
- **Long text** entities → ROUGE scores

In [ ]:
import numpy as np

short_entities = [e for e, c in feature_categories.items() if c == FeatureCategory.SHORT_TEXT]
exact_agg = (
    df_eval[df_eval['entity'].isin(short_entities)]
    .groupby(['model', 'pretty_name'])
    .agg({'exact_match': 'mean', 'cer_score': 'mean'})
    .reset_index()
    .rename(columns={'exact_match': 'accuracy (exact match)'})
)

long_entities = [e for e, c in feature_categories.items() if c == FeatureCategory.LONG_TEXT]
rouge_agg = (
    df_eval[df_eval['entity'].isin(long_entities)]
    .groupby(['model', 'pretty_name'])
    .agg({'rouge1': 'mean', 'rouge2': 'mean', 'rougeL': 'mean', 'rougeLsum': 'mean'})
    .reset_index()
)

aggregated_metrics = rouge_agg.merge(exact_agg, on=['model', 'pretty_name'], how='inner')
metric_cols = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum', 'accuracy (exact match)', 'cer_score']
aggregated_metrics[metric_cols] = aggregated_metrics[metric_cols].round(3)

def highlight_max(s, props=''): return np.where(s == np.nanmax(s.values), props, '')
def highlight_min(s, props=''): return np.where(s == np.nanmin(s.values), props, '')

aggregated_metrics.style \
    .apply(highlight_max, props='background-color:#99d594;', axis=0, subset=['rouge1','rouge2','rougeL','rougeLsum','accuracy (exact match)']) \
    .apply(highlight_min, props='background-color:#99d594;', axis=0, subset=['cer_score'])

## 7 · Visual Diff — Prediction vs Ground Truth

Displays the document image alongside a colour-coded diff between
the predicted JSON and the ground-truth JSON for the first `N_SHOW` samples.

In [ ]:
import textwrap
from PIL import Image
import matplotlib.pyplot as plt

N_SHOW = 15
COL_W  = 35  # column width for GT and Pred

for i, r in enumerate(rows[:N_SHOW]):
    gt_flat   = flatten_dict(r['gt'])
    pred_flat = flatten_dict(r['pred']) if r['valid_json'] else {}

    p = Path(r['image'])
    if not p.is_absolute(): p = IMAGES_DIR / p.name
    try:
        img = Image.open(p).convert('RGB')
        plt.figure(figsize=(5, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"#{i+1} | {r['doc_type']}")
        plt.show()
    except Exception as e:
        print(f"Could not load image: {e}")

    status = '✓ valid JSON' if r['valid_json'] else '✗ invalid JSON'
    print(status)
    print(f"{'Field':<40} {'GT':<{COL_W}} Pred")
    print("-" * (40 + COL_W * 2 + 2))

    for k in sorted(gt_flat.keys()):
        gv = str(gt_flat.get(k, '')).strip()
        pv = str(pred_flat.get(k, '')).strip()
        ok = '✓' if gv == pv else '✗'

        gv_lines = textwrap.wrap(gv, COL_W) or ['']
        pv_lines = textwrap.wrap(pv, COL_W) or ['']
        n_lines  = max(len(gv_lines), len(pv_lines))

        for j in range(n_lines):
            prefix = f"{ok} {k:<38}" if j == 0 else f"  {'':<38}"
            g  = gv_lines[j] if j < len(gv_lines) else ''
            p_ = pv_lines[j] if j < len(pv_lines) else ''
            print(f"{prefix} {g:<{COL_W}} {p_}")

    print()